# Sum-difference problem II

In [ ]:
#@title Verification code

# pylint: disable=unused-import
# pylint: disable=g-bad-import-order



def get_score(best_list):
  """Returns the score for the given list using Numba."""
  if len(best_list) < 2:
    return 0

  # if the list contains non-integers, return 0
  if not all(isinstance(x, int) for x in best_list):
    return 0

  return get_score_numba(best_list) + (1.0 - 1.0 / len(set(best_list))) / 100.0


@njit
def get_score_numba(best_list):
  """Returns the score for the given list using Numba."""

  a_minus_a = set()
  for i in best_list:
    for j in best_list:
      a_minus_a.add(i - j)

  a_plus_a = set()
  for i in best_list:
    for j in best_list:
      a_plus_a.add(i + j)

  lhs = len(a_minus_a)
  rhs = len(a_plus_a)

  try:
    return math.log(lhs) / math.log(rhs)
  except ValueError:
    return 0


def format_feedback_repr(feedback):
  """Formats feedback dictionary for representation in code."""
  formatted_feedback = {}
  for key, value in feedback.items():
    if isinstance(value, np.ndarray):
      repr_str = repr(value)  # Get repr string (e.g., "array([[...], [...]])")
      cleaned_repr_str = re.sub(r'[\n\s]+', ' ', repr_str)  # Clean up

      # Remove the leading "array(" and trailing ")" from repr string, then wrap
      # with "np.array(...)"
      array_content = cleaned_repr_str[
          6:-1
      ]  # Extract content inside "array(...)"

      if np.iscomplexobj(value):
        formatted_feedback[key] = (  # Use extracted content in np.array
            f'np.array({array_content}, dtype=np.complex128)'
        )
      elif not np.issubdtype(value.dtype, np.inexact):
        formatted_feedback[key] = f'np.array({array_content}, dtype=np.float64)'
      else:
        formatted_feedback[key] = f'np.array({array_content})'

    elif isinstance(value, list):
      formatted_feedback[key] = repr(value)  # Use standard repr for lists
    else:
      formatted_feedback[key] = repr(value)
  return formatted_feedback


def evaluate(
    hypers: Mapping[str, Any],
) -> tuple[dict[str, float], dict[str, str]]:
  """Returns the ratio for the Sum-Difference Conjecture using joint distribution."""
  result = {}
  feedback = {}
  del hypers
  best_list = search_for_best_set()
  score = get_score(best_list)

  result['score'] = score
  feedback['best_list'] = best_list
  feedback['best_score_found'] = score
  feedback = format_feedback_repr(feedback)
  return result, feedback

In [ ]:
#@title Initial program

"""Finds joint probability distributions to test Sum-Difference Conjecture."""
import itertools
import logging
import time
from scipy import integrate
import numpy as np
from scipy import optimize
import warnings
import math
import re
from typing import Any, Callable, Mapping, List, Tuple
import scipy.linalg as la
import numpy.polynomial.polynomial as poly
import collections
import numba

njit = numba.njit
minimize = optimize.minimize



best_list_iqhg = [7, 15, 18, 22, -3, -2]



def search_for_best_set() -> List[int]:
  """Searches for joint probability distributions maximizing the ratio."""
  best_list = best_list_iqhg.copy()
  curr_list = best_list.copy()
  best_score = get_score(best_list)
  eval_count = 0
  start_time = time.time()
  while time.time() - start_time < 100:  # Search for 1000 seconds
    # Mutate best construction
    random_index = np.random.randint(0, len(curr_list))
    curr_list[random_index] += np.random.randint(-3, 4)
    score = get_score(curr_list)
    eval_count += 1
    if np.random.rand() < 0.05:
      curr_list.append(np.random.randint(1, 20))
    if np.random.rand() < 0.03 and len(curr_list) > 5:
      curr_list = curr_list[1:]
    if len(curr_list) > 50:
      curr_list = curr_list[10:]
    if score > best_score:
      best_score = score
      best_list = curr_list.copy()
      print(f'Best score: {score}')
      print(set(best_list))
  return best_list

**Prompt used**

Ruzsa's inequality problem

Act as an expert software developer and optimization specialist specializing in
creating python lists of distinct integers with certain properties. Your task is to find a python list A consisting distinct integers wit the property that the set A - A is as large as possible, and at the same time the set A + A is as small as possible.
Here A + A is the set of all possible sums of two elements of A.

The exact score function your construction will be evaluated on is given below:

def get_score(best_list):
  """Returns the score for the given list."""
  # Calculate A - A where A = best_list
  if len(best_list) < 2:
    return 0

best_list_set = set(best_list)
  a_minus_a = set()
  for i in best_list:
    for j in best_list:
      a_minus_a.add(i - j)

a_plus_a = set()
  for i in best_list:
    for j in best_list:
      a_plus_a.add(i + j)

lhs = len(a_minus_a)
  rhs = len(a_plus_a)

if rhs == 1:
    print(best_list)
    print(a_plus_a)

return math.log(lhs) / math.log(rhs)

You may code up any search method you want, and you are allowed to call the
get_score() function as many times as you want. You have
access to it, you don't need to code up the get_score()
function. You want the score it gives you to be as high as possible!

Your task is to write a search function that searches for the best list. Your
function will have 1000 seconds to run, and after that it has to have returned
the best construction it found. If after 1000 seconds it has not returned
anything, it will be terminated with negative infinity points. You can use your
time best if you have an outer loop of the form
"while time.time() - start_time < 1000:"
or similar, just don't forget to define the "start_time" variable early
in your program. You may choose n, the length of the list, to be anything in
this experiment. Larger values of n have more potential for bigger scores, but
the search space is larger so they are harder to find. You'll have to balance it
accordingly!

Expert advice: consider a high-dimensional simplex $A =  (x_1,\dots,x_N) \in \Z_+^N: \sum_i x_i \leq N/2 $.

## What AlphaEvolve found

The best known lower bound is $C \geq \frac{\log(1+\sqrt{2})}{\log 2} = 1.2715\ldots$, coming from a high-dimensional simplex construction. Without any human hints, AlphaEvolve was not able to discover this construction within a few hours, and only managed to find constructions giving a lower bound of around 1.21.